In [9]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

In [10]:
iris = load_iris()
X = iris['data']
y = iris['target']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1./3, random_state=1
)

In [12]:
X_train_norm = (X_train - X_train.mean()) / X_train.std()
X_train_norm = torch.from_numpy(X_train_norm).float()
y_train = torch.from_numpy(y_train)

print('X_train_norm shape:', X_train_norm.shape)
print('y_train shape:', y_train.shape)

X_train_norm shape: torch.Size([100, 4])
y_train shape: torch.Size([100])


In [13]:
model = nn.Sequential(
    nn.Linear(4,16),
    nn.ReLU(),
    nn.Linear(16,3)
)

In [14]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [15]:
print('X shape:', X.shape)
print('y shape:', y.shape)
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_train_norm shape:', X_train_norm.shape)

X shape: (150, 4)
y shape: (150,)
X_train shape: (100, 4)
y_train shape: torch.Size([100])
X_train_norm shape: torch.Size([100, 4])


In [16]:
for epoch in range(100):
    pred = model(X_train_norm)
    loss = loss_fn(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f'epoch {epoch}, loss {loss.item():.4f}')

epoch 0, loss 1.1413
epoch 20, loss 1.0871
epoch 40, loss 1.0331
epoch 60, loss 0.9745
epoch 80, loss 0.9108


In [17]:
X_test_norm = (X_test - X_train.mean()) / X_train.std()
X_test_norm = torch.from_numpy(X_test_norm).float()
y_test = torch.from_numpy(y_test)

In [18]:
pred_test = model(X_test_norm)
predicted_labels = torch.argmax(pred_test, dim=1)

In [19]:
correct = (predicted_labels == y_test).sum()
accuracy = correct / len(y_test)
print("Test accuracy:", accuracy.item())

Test accuracy: 0.6200000047683716


In [20]:
torch.save(model, 'iris_classifier.pt')
model_new = torch.load('iris_classifier.pt', weights_only=False)
pred_test_reloaded = model_new(X_test_norm)
print('Reloaded model works:', torch.equal(pred_test, pred_test_reloaded))

Reloaded model works: True
